In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import rasterio as rio

In [ ]:
from scipy import stats

In [ ]:
import geopandas as gpd

In [ ]:
import pandas as pd

In [ ]:
from shapely import Point
import numpy as np

In [ ]:
import os
from scipy.stats import pearsonr

In [ ]:
out_path=r'C:\Local\Desktop_previous\miscellaneous\Neha\YieldData\21Jan\alldistricts'

In [ ]:
def get_coord(shape):
    shapefile=pd.read_csv(shape)
    coords = [(x,y) for x, y in zip(shapefile.longitude, shapefile.latitude)]
    return coords

In [ ]:
def getRasterValue(image,coords):
    ras = rio.open(image)
    return [x[0] for x in ras.sample(coords)]

In [ ]:
def mape(y_true, y_pred): 
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


In [ ]:
def indexofagreement(y_true, y_pred):
    if len(y_true)>1:
        muo = (y_true).mean()
        n=[]
        din=[]
        for d in range(len(y_true)):
            n.append((y_true[d]-y_pred[d])**2)
            din.append((abs(y_pred[d]-muo)+abs(y_true[d]))**2)
        ioa = sum(n)/sum(din)
    else:
        ioa=1
    return 1-ioa

In [ ]:
def accuracy_parameter(df):
    y_test=np.array(df['Yield (kg_p_ha)'])
    y_pred=np.array(df['Predicted'])
    mae = mean_absolute_error(y_true=y_test,y_pred=y_pred)
    mse = mean_squared_error(y_true=y_test,y_pred=y_pred) #default=True
    rmse = mean_squared_error(y_true=y_test,y_pred=y_pred,squared=False)
    if len(y_test)>2:
        r2 = pearsonr(y_test,y_pred)[0]
    else:
        r2=0
    maper = mape(y_test,y_pred)
    test_mean=y_test.mean()
    pred_mean=y_pred.mean()
    t_stat, p_val = stats.ttest_ind( y_pred,y_test)
    ioa = indexofagreement(y_test,y_pred)
    # print("MAE:",round(mae,2))
    # print("MSE:",round(mse,2))
    # print("RMSE:",round(rmse,2))
    # print("R2:",round(r2,2))
    # print("MAPE:",round(maper,2))
    
    return mae,mse,rmse,r2,maper,t_stat, p_val,ioa,test_mean, pred_mean

In [ ]:
# raster = r'C:\Users\PushkarGaur\Downloads\Osmanabad_bengal_gram_yield\Osmanabad_bengal_gram_yield.tif'
# shapefile = r'C:\Users\PushkarGaur\Downloads\Siddipet GP\Siddipet.shp'
# points = r'C:\Users\PushkarGaur\Downloads\Siddipet_Maize_Rabi2022-23\siddipet.csv'
# shp = gpd.read_file(shapefile)
# pts = pd.read_csv(points)
# geometry = [Point(xy) for xy in zip(pts['longitude'], pts['latitude'])]
# pt_gdf = gpd.GeoDataFrame(pts, crs='EPSG:4326', geometry=geometry)
# shp = shp.to_crs(pt_gdf.crs)
# # pt_gdf['Predicted']=getRasterValue(raster,get_coord(points))
# # pt_gdf=pt_gdf[pt_gdf['Predicted']>0]
# # pt_gdf['difference']=abs(pt_gdf['Predicted']-pt_gdf['Yield(kg_p_ha)'])
# # pt_gdf=pt_gdf[pt_gdf['difference']<500]
# import seaborn as sns
# test_df = gpd.sjoin(pt_gdf,shp,how='inner', predicate='intersects')
# grp = test_df.groupby('GP_Name')
# data=[]
# npts=[]
# fid=[]
# for d in grp:
#     npts.append(d[1].shape[0])
#     data.append(accuracy_parameter(d[1]))
#     fid.append(d[0])

In [ ]:
dist=[]
param=[]
for r,d,f in os.walk(out_path):
    for fl in f:
        if fl.endswith('.xlsx'):
            print(fl)
            df=pd.read_excel(os.path.join(r,fl))
            dist.append(fl.split('.')[0].split('_')[0])
            param.append((accuracy_parameter(df)))

In [ ]:
df=pd.DataFrame(param,columns=['MAE','MSE','RMSE','R','MAPE','T','p','IndexOfAgreement','Observed','Predicted'])
df['District']=dist

In [ ]:
df=df.drop(columns=['MSE'])

In [ ]:
df

In [ ]:
df.to_excel(os.path.join(out_path,'AllDistricts.xlsx'))